# 第 6 章 · Planning（CrewAI）

书里的做法：把"先列计划、再按计划写"写进**一条 task 的两句指令**里，
由同一个 agent 在同一个回合内完成。

⚠️ 注意 `process=Process.sequential` 只决定"任务之间怎么排"，**它不做规划**——
规划藏在 `Task.description` 那两句话里。真正的规划形态（计划成为可见的中间产物）
见 `../../ch06_planning/README.md`。

kernel 选 **`Python (agentic-crewai)`**。

In [1]:
import os
import threading

from crewai import Agent, Task, Crew, Process, LLM

# 【适配】CrewAI 1.15 改用原生 provider：Agent.llm 只接受 str | BaseLLM，不再收 LangChain 对象，
#   且 "openai/xxx" 这种前缀要走 litellm 回落（未安装）。deepseek 是内置 provider，用 deepseek/ 前缀。
llm = LLM(model="deepseek/deepseek-flash",
          api_key=os.environ["OPENAI_API_KEY"],
          base_url="https://api.deepseek.com/v1")

# ★ 本章新增：规划被写成 task 里的两步指令 —— 先列 bullet-point 计划，再按计划写。
#   计划不落盘、不跨回合、不经过审阅，本质是"让模型先打个草稿再动笔"。
planner_writer_agent = Agent(
    role='Article Planner and Writer',
    goal='Plan and then write a concise, engaging summary on a specified topic.',
    backstory=(
        'You are an expert technical writer and content strategist. '
        'Your strength lies in creating a clear, actionable plan before writing, '
        'ensuring the final summary is both informative and easy to digest.'
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

topic = "The importance of Reinforcement Learning in AI"
high_level_task = Task(
    description=(
        f"1. Create a bullet-point plan for a summary on the topic: '{topic}'.\n"
        f"2. Write the summary based on your plan, keeping it around 200 words."
    ),
    expected_output=(
        "A final report containing two distinct sections:\n\n"
        "### Plan\n"
        "- A bulleted list outlining the main points of the summary.\n\n"
        "### Summary\n"
        "- A concise and well-structured summary of the topic."
    ),
    agent=planner_writer_agent,
)

# ★ 本章新增 Process：决定"任务之间怎么排"。sequential = 挨个跑，不是并行、也不是规划。
crew = Crew(
    agents=[planner_writer_agent],
    tasks=[high_level_task],
    process=Process.sequential,
)


# 【适配】kernel 里有运行中的事件循环，同步 kickoff() 会抛 RuntimeError → 放进独立线程（同第 5 章）
def run_in_thread(fn):
    box = {}

    def _target():
        try:
            box["result"] = fn()
        except BaseException as e:               # noqa: BLE001
            box["error"] = e

    th = threading.Thread(target=_target)
    th.start()
    th.join()
    if "error" in box:
        raise box["error"]
    return box["result"]


print("## Running the planning and writing task ##")
result = run_in_thread(crew.kickoff)

print("\n\n---\n## Task Result ##\n---")
print(result)

## Running the planning and writing task ##


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Article Planner and Writer                                                                              │
│                                                                                                                 │
│  Task: 1. Create a bullet-point plan for a summary on the topic: 'The importance of Reinforcement Learning in   │
│  AI'.                                                                                                           │
│  2. Write the summary based on your plan, keeping it around 200 words.                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Article Planner and Writer                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Plan                                                                                                       │
│  - Define Reinforcement Learning (RL) as learning through interaction, rewards, and trial-and-error.            │
│  - Explain why RL matters: it handles sequential decisions where actions affect future outcomes.                │
│  - Highlight RL’s role beyond supervised learning: optimizing long-term goals rather than matching labeled      │
│  data.                                                                                                          │
│  - Show key applications: robotics, game-playing, autonomous systems, recommendation, and resource management.  │
│  - Emphasize RL’s growing importance in modern AI, especially fine-tuning large language models via human       │
│  feedback.                                                                                                      │
│  - Note challenges that shape responsible adoption: sample efficiency, safety, stability, and reward design.    │
│  - Conclude that RL is essential for adaptive, goal-driven AI systems.                                          │
│                                                                                                                 │
│  ### Summary                                                                                                    │
│  Reinforcement Learning (RL) is important in AI because it enables systems to learn by interacting with an      │
│  environment and optimizing for long-term rewards. Unlike supervised learning, which relies on labeled          │
│  examples, RL trains agents through trial and error, making it well suited to sequential decision-making        │
│  problems where today’s action shapes tomorrow’s options. This capability is central to robotics,               │
│  game-playing, autonomous systems, recommendation engines, and resource management. RL also plays a growing     │
│  role in modern generative AI: techniques such as reinforcement learning from human feedback help align large   │
│  language models with human preferences and safety goals. In addition, RL supports adaptive behavior in         │
│  dynamic environments where fixed rules are insufficient. However, its importance comes with challenges,        │
│  including sample inefficiency, unstable training, reward design, and safety risks. Addressing these issues is  │
│  essential for deploying RL reliably in high-stakes settings. Overall, RL is a foundational technology for      │
│  building AI systems that can plan, adapt, and pursue complex goals—making it indispensable for the next        │
│  generation of intelligent, autonomous applications.                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



---
## Task Result ##
---
### Plan
- Define Reinforcement Learning (RL) as learning through interaction, rewards, and trial-and-error.
- Explain why RL matters: it handles sequential decisions where actions affect future outcomes.
- Highlight RL’s role beyond supervised learning: optimizing long-term goals rather than matching labeled data.
- Show key applications: robotics, game-playing, autonomous systems, recommendation, and resource management.
- Emphasize RL’s growing importance in modern AI, especially fine-tuning large language models via human feedback.
- Note challenges that shape responsible adoption: sample efficiency, safety, stability, and reward design.
- Conclude that RL is essential for adaptive, goal-driven AI systems.

### Summary
Reinforcement Learning (RL) is important in AI because it enables systems to learn by interacting with an environment and optimizing for long-term rewards. Unlike supervised learning, which relies on labeled examples, RL trains agents thro

In [2]:
# ===== 实验：persona（role / goal / backstory）到底起了多少作用？=====
# 做法：任务描述、期望输出完全不变，只把 agent 的人设换成"几乎空白"，对比两者输出。
import re

minimal_agent = Agent(
    role='Writer',          # 只留一个身份词
    goal='',                # 清空
    backstory='',           # 清空
    verbose=False,
    allow_delegation=False,
    llm=llm,
)

# task 是绑定 agent 的，所以用同样的描述重建一个
minimal_task = Task(
    description=high_level_task.description,
    expected_output=high_level_task.expected_output,
    agent=minimal_agent,
)

print("## 对照组：几乎没有人设的 agent ##")
minimal_result = run_in_thread(
    Crew(agents=[minimal_agent], tasks=[minimal_task], process=Process.sequential).kickoff
)


def plan_stats(text: str):
    """数 ### Plan 里的 bullet 条数，取 ### Summary 的词数。"""
    m = re.search(r"###\s*Plan(.*?)###\s*Summary", text, re.S | re.I)
    bullets = re.findall(r"^\s*[-*]\s+(.+)$", m.group(1), re.M) if m else []
    s = re.search(r"###\s*Summary(.*)$", text, re.S | re.I)
    body = s.group(1).strip() if s else text
    return bullets, len(body.split())


full_bullets, full_words = plan_stats(str(result))
min_bullets, min_words = plan_stats(str(minimal_result))

print("\n---\n## 对比 ##\n---")
print(f"{'':12s}{'完整人设':>10s}{'几乎无人设':>12s}")
print(f"{'计划条数':12s}{len(full_bullets):>10d}{len(min_bullets):>12d}")
print(f"{'正文词数':12s}{full_words:>10d}{min_words:>12d}")

print("\n=== 完整人设 · 计划 ===")
for i, b in enumerate(full_bullets, 1):
    print(f"  {i}. {b}")
print("\n=== 几乎无人设 · 计划 ===")
for i, b in enumerate(min_bullets, 1):
    print(f"  {i}. {b}")

## 对照组：几乎没有人设的 agent ##



---
## 对比 ##
---
                  完整人设       几乎无人设
计划条数                 7           7
正文词数               165         122

=== 完整人设 · 计划 ===
  1. Define Reinforcement Learning (RL) as learning through interaction, rewards, and trial-and-error.
  2. Explain why RL matters: it handles sequential decisions where actions affect future outcomes.
  3. Highlight RL’s role beyond supervised learning: optimizing long-term goals rather than matching labeled data.
  4. Show key applications: robotics, game-playing, autonomous systems, recommendation, and resource management.
  5. Emphasize RL’s growing importance in modern AI, especially fine-tuning large language models via human feedback.
  6. Note challenges that shape responsible adoption: sample efficiency, safety, stability, and reward design.
  7. Conclude that RL is essential for adaptive, goal-driven AI systems.

=== 几乎无人设 · 计划 ===
  1. Define Reinforcement Learning (RL) as learning through interaction, feedback, and reward maximization